# Optimal transport based dataset similarity

In [ ]:
%env XLA_PYTHON_CLIENT_MEM_FRACTION=.99

In [ ]:
%load_ext autoreload

In [ ]:
import os
import pickle
from os.path import join, isfile

import pandas as pd

In [ ]:
%autoreload
from pairot.dataset_ot import DatasetMapping

## Load data

In [ ]:
DATA_PATH = "/vol/data/dataset-similarity/preprocessed"
CACHE_DIR = "/vol/data/dataset-similarity/cache"

QUERY_DATASET = "7d7cabfd-1d1f-40af-96b7-26a0825a306d"
REF_DATASET = "ced320a1-29f3-47c1-a735-513c7084d508_CAP"
N_TOP_GENES = 750

VERSION = (
    f"{QUERY_DATASET}+{REF_DATASET}"
    "+n_genes_ova=10"
    "+n_genes_ava=3"
    "+tau=1.0"
    "+epsilon=0.05"
    "+overlap_threshold_ava=0.3"
    "+overlap_n_genes_ava=10"
)

In [ ]:
save_dir = join("/vol/data/dataset-similarity/models/similarityOT", VERSION)
fig_dir = join("/vol/data/dataset-similarity/figures/similarityOT", VERSION)
os.makedirs(save_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
cache_file = join(
    CACHE_DIR, 
    "+".join([QUERY_DATASET, REF_DATASET, f"{N_TOP_GENES}HVGs"]) + ".pickle"
)
with open(cache_file, "rb") as f:
    adata_query, adata_ref = pickle.load(f)

In [ ]:
adata_query

In [ ]:
adata_ref

## Initialize and fit optimal transport model

In [ ]:
dataset_map = DatasetMapping(adata_query, adata_ref)

In [ ]:



if not isfile(join(save_dir, f"ot-model-state.pkl")):
    dataset_map.init_geom(
        n_genes_ova=10,
        n_genes_ava=3,
        overlap_threshold_ava=0.3,
        overlap_n_genes_ava=10,
        batch_size=4096,
        epsilon=0.05,
    )
    print(f"x size: {dataset_map.geom.x.shape[0] * dataset_map.geom.x.shape[1] * 4 / 1000**3} GB")
    print(f"y size: {dataset_map.geom.y.shape[0] * dataset_map.geom.y.shape[1] * 4 / 1000**3} GB")
    dataset_map.init_problem(tau_a=1.0, tau_b=1.0)
    dataset_map.solve()
    dataset_map.pickle_state(join(save_dir, f"ot-model-state.pkl"))
else:
    print("Using cached OT model...")
    dataset_map.load_state(join(save_dir, f"ot-model-state.pkl"))

In [ ]:
dataset_map.ot_solution.reg_ot_cost

## Visualize optimal transport mapping

In [ ]:
%autoreload
from pairot.plotting import plot_cluster_mapping, plot_cluster_distance, plot_sankey

### Cluster mapping

In [ ]:
if any([
    not isfile(join(save_dir, f"mapping_{k}.parquet"))
    for k in ["mean", "jensen_shannon", "transported_mass"]
]):
    mappings = dataset_map.compute_cluster_mapping()
    for key, value in mappings.items():
        value.to_parquet(join(save_dir, f"mapping_{key}.parquet"))
    mapping_mean = mappings["mean"]
    mapping_jenson_shannon = mappings["jensen_shannon"]
    mapping_transported_mass = mappings["transported_mass"]
else:
    mapping_mean = pd.read_parquet(join(save_dir, "mapping_mean.parquet"))
    mapping_jenson_shannon = pd.read_parquet(join(save_dir, "mapping_jensen_shannon.parquet"))
    mapping_transported_mass = pd.read_parquet(join(save_dir, "mapping_transported_mass.parquet"))

In [ ]:
fig = plot_cluster_mapping(mapping_mean, show=False)
fig.write_html(join(fig_dir, f"mapping_marginal_contribution.html"))
fig = plot_cluster_mapping(mapping_jenson_shannon, show=False)
fig.write_html(join(fig_dir, f"mapping_jenson_shannon.html"))
fig = plot_cluster_mapping(mapping_transported_mass, show=False)
fig.write_html(join(fig_dir, f"mapping_transported_mass.html"))

In [ ]:
plot_cluster_mapping(mapping_mean, show=True)

### Cluster distances

In [ ]:
if not isfile(join(save_dir, "distance.parquet")):
    distance = dataset_map.compute_cluster_distances(n_samples=25000)
    distance.to_parquet(join(save_dir, "distance.parquet"))
else:
    distance = pd.read_parquet(join(save_dir, "distance.parquet"))

In [ ]:
plot_cluster_distance(distance, show=True)

In [ ]:
fig = plot_cluster_distance(distance, show=False)
fig.write_html(join(fig_dir, f"distance.html"))

### Sankey visualization

In [ ]:
plot_sankey(mapping_mean, distance, filter_threshold=0.25)

In [ ]:
fig = plot_sankey(mapping_mean, distance, show=False)
fig.write_html(join(fig_dir, f"sankey.html"))

### Top n most similar clusters

In [ ]:
top_n_labels = DatasetMapping.select_most_similar_clusters(
    mapping_mean, 
    distance, 
    threshold_mapping=0.25,
    threshold_distance=1., 
    n_top=None
)
top_n_labels